# Importing libraries and loading data

In [43]:
#Import libraries and load data
import pandas as pd
import numpy as np

In [44]:
transactions = pd.read_csv('trans_labelled_clean.csv')
cards = pd.read_csv('cards_data_south_africa.csv')
users = pd.read_csv('user_data_south_africa.csv')

trans_merged = pd.merge(transactions, cards, left_on='card_id', right_on='id')
trans_merged = pd.merge(trans_merged, users, left_on='client_id_x', right_on='id')
trans_merged = trans_merged.drop(columns=['id_y', 'client_id_y', 'id'])
trans_merged = trans_merged.rename(columns={
    'id_x': 'id',
    'client_id_x': 'user_id',
})

# Amount Feature

In [45]:
trans_merged['is_negative_amount'] = trans_merged['amount'] < 0
trans_merged['abs_amount'] = trans_merged['amount'].abs()
trans_merged['is_negative_amount'] = trans_merged['is_negative_amount'].astype(int)
trans_merged['is_fraud'] = trans_merged['is_fraud'].astype(int)

# Time-Based Feature

In [46]:
from datetime import datetime, timedelta


trans_merged["date"] = pd.to_datetime(trans_merged["date"])
trans_merged = trans_merged.sort_values(['user_id', 'date']).reset_index(drop=True)
trans_merged["day_of_week"] = trans_merged["date"].dt.dayofweek
trans_merged["trans_hour"] = trans_merged["date"].dt.hour
trans_merged["trans_weekend"] = trans_merged["day_of_week"].isin([5, 6]).astype(int)
trans_merged["trans_day"] = trans_merged["date"].dt.day
trans_merged["trans_month"] = trans_merged["date"].dt.month
trans_merged["trans_year"] = trans_merged["date"].dt.year

# Outliers and Z-Score

In [47]:
trans_merged['date'] = pd.to_datetime(trans_merged['date'])
trans_merged = trans_merged.sort_values('date').reset_index(drop=True)

def iqr_outlier_mask(s: pd.Series, k: float = 1.5):
    q1, q3 = s.quantile([0.25, 0.75])
    iqr = q3 - q1
    lower_bound = q1 - k * iqr
    upper_bound = q3 + k * iqr
    return (s < lower_bound) | (s > upper_bound), lower_bound, upper_bound

mask_iqr, lower_iqr, upper_iqr = iqr_outlier_mask(trans_merged['abs_amount'], k=1.5)
df_iqr_outliers = trans_merged.loc[mask_iqr, ["abs_amount"]]
df_iqr_outliers.head(40).sort_values(by='abs_amount')

,abs_amount
785,2880.00
852,3002.40
450,3054.78
944,3133.62
354,3192.84
699,3234.96
832,3311.64
224,3600.00
180,3739.32
278,3768.48


In [48]:
def zscore_outlier_mask(s: pd.Series, threshold: float = 3.5):
    z = (s - s.mean()) / s.std()
    mask = z.abs() > threshold
    return mask, z

mask_z, z = zscore_outlier_mask(trans_merged['abs_amount'], threshold=3.0)
df_z_outliers = trans_merged.loc[mask_z, ["abs_amount"]]
df_z_outliers.head(30).sort_values(by='abs_amount')

,abs_amount
1374,5038.56
714,5153.58
1811,5497.74
971,5598.00
1803,5724.00
229,5778.00
857,5958.00
1256,5970.78
646,6096.60
331,6156.00


In [49]:
trans_merged['is_high_amount'] = mask_iqr.astype(int)
trans_merged['is_zscore_outlier'] = mask_z.astype(int)
trans_merged['monthly_income'] = trans_merged['yearly_income'] / 12
trans_merged["amt_to_income_ratio"] = (trans_merged["abs_amount"] / trans_merged["yearly_income"]) * 100

# Time-Window Features

In [50]:
def build_time_window_features(
    trans_merged,
    ts_col="date",
    amount_col="abs_amount"
):
    # ---- mandatory ordering ----
    trans_merged = trans_merged.sort_values(["user_id", ts_col]).reset_index(drop=True)

    # ============================
    # USER-LEVEL FEATURES
    # ============================
    g_user = trans_merged.groupby("user_id", group_keys=False)

    trans_merged["user_txn_count_24h"] = (
        g_user.rolling("24h", on=ts_col)[amount_col]
        .count()
        .shift(1)
        .reset_index(drop=True)
    )

    trans_merged["user_txn_count_7d"] = (
        g_user.rolling("7d", on=ts_col)[amount_col]
        .count()
        .shift(1)
        .reset_index(drop=True)
    )

    trans_merged["user_amt_sum_24h"] = (
        g_user.rolling("24h", on=ts_col)[amount_col]
        .sum()
        .shift(1)
        .reset_index(drop=True)
    )

    # ---- stronger than raw sums ----
    trans_merged["user_avg_amt_24h"] = (
        trans_merged["user_amt_sum_24h"] /
        (trans_merged["user_txn_count_24h"] + 1)
    )

    # ============================
    # MERCHANT-LEVEL FEATURES
    # ============================
    trans_merged = trans_merged.sort_values(["merchant_id", ts_col]).reset_index(drop=True)
    g_merchant = trans_merged.groupby("merchant_id", group_keys=False)

    trans_merged["merchant_txn_count_24h"] = (
        g_merchant.rolling("24h", on=ts_col)[amount_col]
        .count()
        .shift(1)
        .reset_index(drop=True)
    )

    # ============================
    # CLEANUP
    # ============================
    trans_merged.replace([np.inf, -np.inf], np.nan, inplace=True)

    return trans_merged

In [51]:
#Applied
trans_merged = build_time_window_features(
    trans_merged,
    ts_col="date",
    amount_col="abs_amount"
)

In [52]:
trans_merged['user_mcc_seen_before'] = (
    trans_merged.sort_values(['user_id', 'date'])
      .groupby(['user_id', 'mcc'])
      .cumcount()
)

trans_merged['is_first_time_user_mcc'] = (trans_merged['user_mcc_seen_before'] == 0).astype(int)

In [53]:
#by merchant
trans_merged['user_merchant_seen_before'] = trans_merged.groupby(['user_id','merchant_id']).cumcount()
trans_merged['is_first_time_user_merchant'] = (trans_merged['user_merchant_seen_before'] == 0).astype(int)

#by city
trans_merged['user_city_seen_before'] = trans_merged.groupby(['user_id','merchant_city']).cumcount()
trans_merged['is_first_time_user_city'] = (trans_merged['user_city_seen_before'] == 0).astype(int)

#by hour
trans_merged['user_hour_seen_before'] = trans_merged.groupby(['user_id','trans_hour']).cumcount()
trans_merged['is_first_time_user_hour'] = (trans_merged['user_hour_seen_before'] == 0).astype(int)

# Geo Features

In [54]:
trans_merged['user_city'] = trans_merged['address'].str.split(',').str[1].str.strip()

city_to_region = {
    "cape town": "Western Cape",
    "pretoria": "Gauteng",
    "johannesburg": "Gauteng",
    "durban": "KwaZulu-Natal",
    "pietermaritzburg": "KwaZulu-Natal",
    "gqeberha": "Eastern Cape",
    "east london": "Eastern Cape",
    "bloemfontein": "Free State",
}

# Normalization
def normalize_city(s):
    if pd.isna(s): return None
    return (str(s).strip().lower()
            .replace('.', '')
            .replace('-', ' ')
            .replace(',', ''))

# Apply
trans_merged['user_city'] = trans_merged['user_city'].map(normalize_city)
trans_merged['user_region'] = trans_merged['user_city'].map(city_to_region).fillna("unknown")

In [55]:
'''trans_merged['user_region'] = trans_merged['user_region'].map(normalize_city)
trans_merged['is_far_from_home'] = (
    (trans_merged['use_chip'] != 'online transaction') &
    (trans_merged['user_region'] != trans_merged['merchant_state'])
).astype(int)'''

"trans_merged['user_region'] = trans_merged['user_region'].map(normalize_city)\ntrans_merged['is_far_from_home'] = (\n    (trans_merged['use_chip'] != 'online transaction') &\n    (trans_merged['user_region'] != trans_merged['merchant_state'])\n).astype(int)"

In [56]:
# ---- Same city ----
trans_merged["is_same_city"] = (
    (trans_merged["user_city"].notnull()) &
    (trans_merged["merchant_city"].notnull()) &
    (trans_merged["user_city"] == trans_merged["merchant_city"])
).astype(int)

# ---- Same region ----
trans_merged["is_same_region"] = (
    (trans_merged["user_region"].notnull()) &
    (trans_merged["merchant_state"].notnull()) &
    (trans_merged["user_region"] == trans_merged["merchant_state"])
).astype(int)

# ---- Different region ----
trans_merged["is_different_region"] = (
    (trans_merged["is_same_region"] == 0) &
    trans_merged["merchant_state"].notnull() &
    trans_merged["user_region"].notnull()
).astype(int)

# Missing Features

In [57]:
trans_merged = trans_merged.sort_values("date")

user_stats = (
    trans_merged.groupby("user_id")["abs_amount"]
      .expanding()
      .agg(["mean", "std"])
      .shift(1)
      .reset_index(level=0, drop=True)
)

trans_merged["z_amount_user"] = (
    (trans_merged["abs_amount"] - user_stats["mean"]) /
    (user_stats["std"].fillna(0) + 1e-6)
)

In [58]:
user_median = (
    trans_merged.groupby("user_id")["abs_amount"]
      .expanding()
      .median()
      .shift(1)
      .reset_index(level=0, drop=True)
)

trans_merged["amount_user_median_ratio"] = trans_merged["abs_amount"] / (user_median + 1e-6)

In [59]:
# Ensure chronological order
trans_merged = trans_merged.sort_values(['user_id', 'date'])

# Cumulative mean hour BEFORE current transaction
trans_merged['user_hour_mean'] = (
    trans_merged.groupby('user_id')['trans_hour']
    .expanding()
    .mean()
    .shift(1)
    .reset_index(level=0, drop=True)
)


# Circular difference
def circular_diff(a, b):
    return np.minimum(abs(a-b), 24 - abs(a-b))

trans_merged['hour_deviation_user'] = circular_diff(
    trans_merged['trans_hour'],
    trans_merged['user_hour_mean']
)

In [60]:
trans_merged = trans_merged.sort_values(["user_id", "date"])

trans_merged["time_diff"] = (
    trans_merged.groupby("user_id")["date"].diff().dt.total_seconds()
)

trans_merged["avg_gap_last3"] = (
    trans_merged.groupby("user_id")["time_diff"]
    .rolling(3)
    .mean()
    .shift(1)
    .reset_index(level=0, drop=True)
)

In [61]:
user_hist = (
    trans_merged.groupby("user_id")["is_fraud"]
      .expanding()
      .mean()
      .shift(1)
      .reset_index(level=0, drop=True)
)

trans_merged["user_hist_fraud_rate"] = user_hist

In [62]:
trans_merged = trans_merged.sort_values(["user_id", "date"])

# Keep fraud dates, otherwise NaT
fraud_date = trans_merged["date"].where(trans_merged["is_fraud"].eq(1))

# For each user, carry the last seen fraud date forward
last_fraud_date = fraud_date.groupby(trans_merged["user_id"]).ffill()

# Shift so that on a fraud row, "last fraud" means the previous one (not itself)
last_fraud_date = last_fraud_date.groupby(trans_merged["user_id"]).shift(1)

trans_merged["days_since_last_fraud"] = (
    (trans_merged["date"] - last_fraud_date).dt.days
)


trans_merged["days_since_last_fraud"] = (
    trans_merged["days_since_last_fraud"]
    .clip(lower=0)
    .fillna(9999)
)


In [63]:
trans_merged.loc[(trans_merged["user_id"].eq(100)) & (trans_merged["is_fraud"] == 1),
                 ["user_id","date","is_fraud","days_since_last_fraud"]].head(50)

,user_id,date,is_fraud,days_since_last_fraud
2386623,100,2022-03-27 12:12:02,1,9999.0
2392812,100,2022-04-15 19:30:45,1,19.0
8376209,100,2022-05-10 08:58:17,1,24.0
2404466,100,2022-05-18 18:33:46,1,8.0
3093660,100,2022-05-20 04:50:06,1,1.0
6397390,100,2022-06-16 18:32:12,1,27.0
7200791,100,2022-07-01 23:15:07,1,15.0
2838763,100,2022-07-02 06:48:07,1,0.0
2970406,100,2022-07-06 00:39:03,1,3.0
7527086,100,2022-09-09 21:54:05,1,65.0


In [64]:
trans_merged["hour_sin"] = np.sin(2 * np.pi * trans_merged["trans_hour"] / 24)
trans_merged["hour_cos"] = np.cos(2 * np.pi * trans_merged["trans_hour"] / 24)

In [65]:
features_no_history = [
    'z_amount_user', 'amount_user_median_ratio',
    'hour_deviation_user', 'user_hour_mean',
    'avg_gap_last3', 'user_hist_fraud_rate', 'user_txn_count_24h',
    'user_txn_count_7d', 'user_amt_sum_24h', 'user_avg_amt_24h',
    'merchant_txn_count_24h']

trans_merged[features_no_history] = trans_merged[features_no_history].fillna(0)
trans_merged.isna().sum().sort_values(ascending=False).head(70)

time_diff                   1219
id                             0
user_id                        0
date                           0
amount                         0
                            ... 
is_different_region            0
z_amount_user                  0
is_same_city                   0
amount_user_median_ratio       0
user_hour_mean                 0
Length: 70, dtype: int64

In [66]:
trans_merged["is_first_tx_user"] = trans_merged.groupby("user_id")["date"].rank(method="first") == 1
trans_merged["is_first_tx_user"] = trans_merged["is_first_tx_user"].astype(int)

trans_merged["time_diff"] = trans_merged["time_diff"].fillna(999999)

In [67]:
trans_merged.isna().sum().sort_values(ascending=False)

id                       0
date                     0
user_id                  0
card_id                  0
amount                   0
                        ..
user_hist_fraud_rate     0
days_since_last_fraud    0
hour_sin                 0
hour_cos                 0
is_first_tx_user         0
Length: 77, dtype: int64

# Behavioral Features

In [68]:
# Sort to ensure proper chronological order
trans_merged = trans_merged.sort_values(["user_id", "date"])

# Count prior uses of this brand by this user
brand_prior_count = (
    trans_merged.groupby(['user_id', 'card_brand'])
    .cumcount()
)

# If prior count = 0 → brand is new to this user
trans_merged['is_unusual_brand_for_user'] = (brand_prior_count == 0).astype(int)

# But ignore first few transactions of the user (too little history)
user_txn_counts = trans_merged.groupby('user_id').cumcount()

trans_merged.loc[user_txn_counts < 3, 'is_unusual_brand_for_user'] = 0

In [69]:
# Previous transaction amount
trans_merged["prev_amount"] = trans_merged.groupby("user_id")["abs_amount"].shift(1)

trans_merged["amount_delta_last_txn"] = trans_merged["abs_amount"] - trans_merged["prev_amount"]
trans_merged["amount_pct_change"] = (trans_merged["abs_amount"] + 1) / (trans_merged["prev_amount"] + 1)

# Rolling mean (last 3)
trans_merged["mean_last3_amount"] = (
    trans_merged.groupby("user_id")["abs_amount"]
      .shift(1)
      .rolling(3)
      .mean()
)

trans_merged["amount_vs_last3_mean"] = (trans_merged["abs_amount"] + 1) / (trans_merged["mean_last3_amount"] + 1)

trans_merged["prev_amount"] = trans_merged["prev_amount"].fillna(trans_merged["abs_amount"])
trans_merged["mean_last3_amount"] = trans_merged["mean_last3_amount"].fillna(trans_merged["abs_amount"])

In [70]:
# Ensure date column is a proper datetime
trans_merged["date"] = pd.to_datetime(trans_merged["date"], errors="coerce")

# Sort chronologically (critical!)
trans_merged = trans_merged.sort_values(["user_id", "date"])

# Previous timestamp per user
trans_merged["prev_time"] = trans_merged.groupby("user_id")["date"].shift(1)

# True time-difference in seconds
trans_merged["time_diff_sec"] = (
    trans_merged["date"] - trans_merged["prev_time"]
).dt.total_seconds()

In [71]:
trans_merged["txn_rate_1h_vs_24h"] = (
    trans_merged["user_txn_count_24h"] /
    (trans_merged["user_txn_count_7d"] + 1e-6))

trans_merged["txn_rate_24h_vs_7d"] = (
    trans_merged["user_txn_count_24h"] /
    (trans_merged["user_txn_count_7d"] + 1e-6))

In [72]:
trans_merged["gap"] = trans_merged["time_diff_sec"]

trans_merged["mean_gap_last5"] = (
    trans_merged.groupby("user_id")["gap"]
    .shift(1)
    .rolling(5)
    .mean()
    .reset_index(level=0, drop=True)
)

trans_merged["std_gap_last5"] = (
    trans_merged.groupby("user_id")["gap"]
    .shift(1)
    .rolling(5)
    .std()
    .reset_index(level=0, drop=True)
)

In [73]:
trans_merged.columns

Index(['id', 'date', 'user_id', 'card_id', 'amount', 'use_chip', 'merchant_id',
       'merchant_city', 'merchant_state', 'zip', 'mcc', 'errors', 'is_fraud',
       'description', 'card_brand', 'card_type', 'card_number', 'expires',
       'cvv', 'has_chip', 'num_cards_issued', 'credit_limit', 'acct_open_date',
       'year_pin_last_changed', 'card_on_dark_web', 'current_age',
       'retirement_age', 'birth_year', 'birth_month', 'gender', 'address',
       'per_capita_income', 'yearly_income', 'total_debt', 'credit_score',
       'num_credit_cards', 'is_negative_amount', 'abs_amount', 'day_of_week',
       'trans_hour', 'trans_weekend', 'trans_day', 'trans_month', 'trans_year',
       'is_high_amount', 'is_zscore_outlier', 'monthly_income',
       'amt_to_income_ratio', 'user_txn_count_24h', 'user_txn_count_7d',
       'user_amt_sum_24h', 'user_avg_amt_24h', 'merchant_txn_count_24h',
       'user_mcc_seen_before', 'is_first_time_user_mcc',
       'user_merchant_seen_before', 'is_first

In [74]:
trans_merged["first_time_and_high_amount"] = (
    (trans_merged["is_first_time_user_merchant"] == 1) &
    (trans_merged["is_zscore_outlier"] > 2)
).astype(int)

trans_merged["first_time_and_far"] = (
    (trans_merged["is_first_time_user_merchant"] == 1) &
    (trans_merged["is_different_region"] == 1)
).astype(int)

trans_merged["first_time_and_night"] = (
    (trans_merged["is_first_time_user_merchant"] == 1) &
    (trans_merged["trans_hour"].between(0, 5))
).astype(int)

In [75]:
# Sort properly
'''trans_merged = trans_merged.sort_values(["user_id", "date"])

# Compute rolling 7-day fraud count per user
trans_merged["user_fraud_count_7d"] = (
    trans_merged
    .groupby("user_id")
    .apply(
        lambda g: g.set_index("date")["is_fraud"]
                    .shift(1)
                    .rolling("7D")
                    .sum()
    )
    .reset_index(level=0, drop=True)
)

# Days since last fraud
last_fraud_time = (
    trans_merged["date"]
    .where(trans_merged["is_fraud"] == 1)
)

last_fraud_time = last_fraud_time.groupby(trans_merged["user_id"]).ffill()

trans_merged["fraud_recency_score"] = np.exp(
    -trans_merged["days_since_last_fraud"] / 7
)'''


'trans_merged = trans_merged.sort_values(["user_id", "date"])\n\n# Compute rolling 7-day fraud count per user\ntrans_merged["user_fraud_count_7d"] = (\n    trans_merged\n    .groupby("user_id")\n    .apply(\n        lambda g: g.set_index("date")["is_fraud"]\n                    .shift(1)\n                    .rolling("7D")\n                    .sum()\n    )\n    .reset_index(level=0, drop=True)\n)\n\n# Days since last fraud\nlast_fraud_time = (\n    trans_merged["date"]\n    .where(trans_merged["is_fraud"] == 1)\n)\n\nlast_fraud_time = last_fraud_time.groupby(trans_merged["user_id"]).ffill()\n\ntrans_merged["fraud_recency_score"] = np.exp(\n    -trans_merged["days_since_last_fraud"] / 7\n)'

In [76]:
trans_merged.columns

Index(['id', 'date', 'user_id', 'card_id', 'amount', 'use_chip', 'merchant_id',
       'merchant_city', 'merchant_state', 'zip', 'mcc', 'errors', 'is_fraud',
       'description', 'card_brand', 'card_type', 'card_number', 'expires',
       'cvv', 'has_chip', 'num_cards_issued', 'credit_limit', 'acct_open_date',
       'year_pin_last_changed', 'card_on_dark_web', 'current_age',
       'retirement_age', 'birth_year', 'birth_month', 'gender', 'address',
       'per_capita_income', 'yearly_income', 'total_debt', 'credit_score',
       'num_credit_cards', 'is_negative_amount', 'abs_amount', 'day_of_week',
       'trans_hour', 'trans_weekend', 'trans_day', 'trans_month', 'trans_year',
       'is_high_amount', 'is_zscore_outlier', 'monthly_income',
       'amt_to_income_ratio', 'user_txn_count_24h', 'user_txn_count_7d',
       'user_amt_sum_24h', 'user_avg_amt_24h', 'merchant_txn_count_24h',
       'user_mcc_seen_before', 'is_first_time_user_mcc',
       'user_merchant_seen_before', 'is_first

In [77]:
trans_merged = trans_merged.sort_values(["user_id", "date"])
trans_merged["prev_city"] = trans_merged.groupby("user_id")["user_city"].shift(1)
trans_merged["city_changed"] = (trans_merged["user_city"] != trans_merged["prev_city"]).astype(int)

trans_merged["time_diff_sec"] = (
    trans_merged["date"] - trans_merged.groupby("user_id")["date"].shift(1)
).dt.total_seconds()

# Changing City Features
trans_merged["city_change_fast"] = (
    (trans_merged["city_changed"] == 1) &
    (trans_merged["time_diff_sec"] < 3600)
).astype(int)

# Night City Change
trans_merged["night_city_change"] = (
    (trans_merged["city_changed"] == 1) &
    (trans_merged["is_first_time_user_hour"] == 1)
).astype(int)

In [78]:
trans_merged["night_x_velocity"] = (
    trans_merged["is_first_time_user_hour"] *
    trans_merged["txn_rate_1h_vs_24h"]
)

In [79]:
trans_merged.isnull().sum().sort_values(ascending=False).head(20)

std_gap_last5            7314
mean_gap_last5           7314
amount_vs_last3_mean     3657
prev_time                1219
time_diff_sec            1219
amount_delta_last_txn    1219
gap                      1219
amount_pct_change        1219
prev_city                1219
merchant_state              0
user_id                     0
card_id                     0
amount                      0
use_chip                    0
merchant_id                 0
merchant_city               0
card_number                 0
expires                     0
cvv                         0
has_chip                    0
dtype: int64

In [81]:
trans_merged["gap"].fillna(999999, inplace=True)
trans_merged["time_diff_sec"].fillna(999999, inplace=True)
trans_merged["mean_gap_last5"].fillna(999999, inplace=True)
trans_merged["std_gap_last5"].fillna(0, inplace=True)   # STD of 1 point = 0
trans_merged["prev_time"] = trans_merged["prev_time"].fillna(trans_merged["date"])
trans_merged["prev_city"] = trans_merged["prev_city"].fillna(trans_merged["user_city"])

In [82]:
trans_merged["amount_delta_last_txn"] = (
    trans_merged["amount_delta_last_txn"]
    .fillna(0)
)

trans_merged["amount_pct_change"] = (
    trans_merged["amount_pct_change"]
    .fillna(1)    # 100% of itself
)

trans_merged["amount_vs_last3_mean"] = (
    trans_merged["amount_vs_last3_mean"]
    .fillna(1)    # self / self
)

In [83]:
trans_merged.shape

(8615533, 98)

In [84]:
cols_to_drop = [
    # IDs / PII / unhelpful identifiers
    'id', 'card_id', 'card_number', 'address', 'zip',

    # Static personal attributes (good for credit scoring, weak for fraud)
    'gender', 'birth_year', 'birth_month', 'retirement_age', 'current_age',
    'num_credit_cards', 'credit_score', 'total_debt', 'yearly_income',
    'monthly_income', 'credit_limit', 'acct_open_date',

    # Sensitive card details
    'cvv', 'expires', 'has_chip', 'card_type', 'description',

    # Redundant merchant / location info (you have better engineered versions)
    'merchant_state', 'mcc', 'user_region'

    # Weak date breakdowns / redundant time features
    'trans_year', 'trans_month', 'trans_day', 'day_of_week', 'trans_weekend', 'trans_hour'

    # Raw boolean/noise fields
    'is_negative_amount', 'is_high_amount',

    # Deprecated or redundant engineered features
    'amount_to_income_ratio',

    # Other weak or redundant fields you mentioned
    'num_cands_issued', 'card_on_dark_web', 'merchant_txn_count_24h',

    # Raw history flags where engineered "first time" features exist
    'user_city_seen_before', 'user_mcc_seen_before', 'user_merchant_seen_before',

    # Potential leakage or inconsistent fields
    'errors']

# Safety: drop only columns that actually exist in the dataframe
cols_to_drop = [c for c in cols_to_drop if c in trans_merged.columns]

print(f"Dropping {len(cols_to_drop)} columns:", cols_to_drop)

# Drop in-place (or assign back if you prefer)
trans_merged = trans_merged.drop(columns=cols_to_drop)

Dropping 35 columns: ['id', 'card_id', 'card_number', 'address', 'zip', 'gender', 'birth_year', 'birth_month', 'retirement_age', 'current_age', 'num_credit_cards', 'credit_score', 'total_debt', 'yearly_income', 'monthly_income', 'credit_limit', 'acct_open_date', 'cvv', 'expires', 'has_chip', 'card_type', 'description', 'merchant_state', 'mcc', 'trans_month', 'trans_day', 'day_of_week', 'trans_weekend', 'is_high_amount', 'card_on_dark_web', 'merchant_txn_count_24h', 'user_city_seen_before', 'user_mcc_seen_before', 'user_merchant_seen_before', 'errors']


In [ ]:
trans_merged.columns

Index(['date', 'user_id', 'amount', 'use_chip', 'merchant_id', 'merchant_city',
       'is_fraud', 'card_brand', 'num_cards_issued', 'year_pin_last_changed',
       'per_capita_income', 'is_negative_amount', 'abs_amount', 'trans_hour',
       'trans_year', 'is_zscore_outlier', 'amt_to_income_ratio',
       'user_txn_count_24h', 'user_txn_count_7d', 'user_amt_sum_24h',
       'user_avg_amt_24h', 'is_first_time_user_mcc',
       'is_first_time_user_merchant', 'is_first_time_user_city',
       'user_hour_seen_before', 'is_first_time_user_hour', 'user_city',
       'city_norm', 'user_region', 'is_same_city', 'is_same_region',
       'is_different_region', 'z_amount_user', 'amount_user_median_ratio',
       'user_hour_mean', 'hour_deviation_user', 'time_diff', 'avg_gap_last3',
       'user_hist_fraud_rate', 'days_since_last_fraud',
       'days_since_last_fraud_user', 'hour_sin', 'hour_cos',
       'is_first_tx_user', 'is_unusual_brand_for_user', 'prev_amount',
       'amount_delta_last_txn

In [85]:
cols_to_drop2 = ['merchant_city',
'user_city',
'user_region',
'trans_hour',
'trans_year',
'time_diff',
'avg_gap_last3',
'num_cards_issued',
'year_pin_last_changed',
'per_capita_income',
'is_negative_amount',
'amt_to_income_ratio',
'prev_time']

cols_to_drop2 = [c for c in cols_to_drop2 if c in trans_merged.columns]

print(f"Dropping {len(cols_to_drop2)} columns:", cols_to_drop2)

# Drop in-place (or assign back if you prefer)
trans_merged = trans_merged.drop(columns=cols_to_drop2)

Dropping 13 columns: ['merchant_city', 'user_city', 'user_region', 'trans_hour', 'trans_year', 'time_diff', 'avg_gap_last3', 'num_cards_issued', 'year_pin_last_changed', 'per_capita_income', 'is_negative_amount', 'amt_to_income_ratio', 'prev_time']


In [86]:
trans_merged.shape

(8615533, 50)

In [87]:
trans_merged.columns

Index(['date', 'user_id', 'amount', 'use_chip', 'merchant_id', 'is_fraud',
       'card_brand', 'abs_amount', 'is_zscore_outlier', 'user_txn_count_24h',
       'user_txn_count_7d', 'user_amt_sum_24h', 'user_avg_amt_24h',
       'is_first_time_user_mcc', 'is_first_time_user_merchant',
       'is_first_time_user_city', 'user_hour_seen_before',
       'is_first_time_user_hour', 'is_same_city', 'is_same_region',
       'is_different_region', 'z_amount_user', 'amount_user_median_ratio',
       'user_hour_mean', 'hour_deviation_user', 'user_hist_fraud_rate',
       'days_since_last_fraud', 'hour_sin', 'hour_cos', 'is_first_tx_user',
       'is_unusual_brand_for_user', 'prev_amount', 'amount_delta_last_txn',
       'amount_pct_change', 'mean_last3_amount', 'amount_vs_last3_mean',
       'time_diff_sec', 'txn_rate_1h_vs_24h', 'txn_rate_24h_vs_7d', 'gap',
       'mean_gap_last5', 'std_gap_last5', 'first_time_and_high_amount',
       'first_time_and_far', 'first_time_and_night', 'prev_city',
   

In [88]:
trans_merged.to_csv("trans_merged.csv", index=False)